In [1]:
import ROOT
from fitHelper import fit, build_sim_ws, Plotter

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x9622920
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x99673e0


In [2]:
ROOT.EnableImplicitMT(6)

In [3]:
# run_ab = 2
# run_ab = 3
# run_ab = 11.2
# split_hel = False
split_hel = True
# constant_parameters = ["epol", "ppol"]
constant_parameters = []
# LCVision quoting 0904.0122
pol_constraint = 2.5e-3

lumi_list = [2, 3, 11.2]

plot_dir = f"plots/fit_results_multi_run{'_split_hel' if split_hel else ''}{'_const_pol' if constant_parameters else ''}{'_pol_constraint' if pol_constraint else ''}"

In [4]:
# oo_names = [
#     "mlvec_reco",
#     "reco",
#     "mlvec_reco_jm",
#     "reco_jm",
#     "mlvec_clean_reco",
#     "clean_reco",
#     "kinfit_clean_reco",
#     "mlvec_clean_reco_jm",
#     "clean_reco_jm",
#     "kinfit_clean_reco_jm",
#     "mlvec_cheat_clean_reco",
#     "cheat_clean_reco",
#     "mlvec_cheat_clean_reco_jm",
#     "cheat_clean_reco_jm",
#     "mc",
#     "nomb_mc",
#     "nomb_mc_rlep",
#     "nomb_mc_rlep_gamma",
#     "nomb_mc_rlep_brems",
#     "nomb_mc_rlep_cheated_brems",
#     "av_mc",
#     "av_nomb_mc",
#     "av_nomb_mc_rlep",
#     "av_nomb_mc_rlep_gamma",
#     "av_nomb_mc_rlep_cheated_brems",
# ]
oo_names = [
    "mlvec_reco",
    "reco",
    "mlvec_reco_jm",
    "reco_jm",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
    # "kinfit_clean_reco",
    "mlvec_clean_reco_jm",
    "clean_reco_jm",
    "clean_brems_reco_jm",
    "mlvec_clean_brems_reco_jm",
    # "kinfit_clean_reco_jm",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    "mlvec_cheat_clean_reco_jm",
    "cheat_clean_reco_jm",
    "mc",
    "nomb_mc",
    "nomb_mc_rlep",
    "nomb_mc_rlep_gamma",
    "nomb_mc_rlep_brems",
    "nomb_mc_rlep_cheated_brems",
    "nurec_nomb_mc",
    "nurec_nomb_mc_rlep",
    "nurec_nomb_mc_rlep_gamma",
    "nurec_nomb_mc_rlep_brems",
    "nurec_nomb_mc_rlep_cheated_brems",
    "nurec_post94_mc",
    "av_mc",
    "av_nomb_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc_rlep_gamma",
    "av_nomb_mc_rlep_brems",
    "av_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_post94_mc",
    "av_nurec_nomb_mc_cc10",
    "mlvec_clean_brems_reco_cc10",
    "mlvec_clean_brems_reco_san",
]
# input_path = "fit-configs/signal-only-mW-pol"
# input_path = "fit-configs/signal-only-mW-pol-new"
input_path = "fit-configs/signal-only-mW-pol-new2"
oo_name = "mlvec_clean_reco"
run_configurations = {
    "new_share_high_ppol" : [
    # (lumi_share, e_pol, p_pol)
        (0.05, -0.8, -0.6),
        (0.45, -0.8, +0.6),
        (0.45, +0.8, -0.6),
        (0.05, +0.8, +0.6),
    ],
    "old_share_high_ppol" : [
        (0.05, -0.8, -0.6),
        (0.675, -0.8, +0.6),
        (0.225, +0.8, -0.6),
        (0.05, +0.8, +0.6),
    ],
    "new_share_low_ppol" : [
        (0.05, -0.8, -0.3),
        (0.45, -0.8, +0.3),
        (0.45, +0.8, -0.3),
        (0.05, +0.8, +0.3),
    ],
    "old_share_low_ppol" : [
        (0.05, -0.8, -0.3),
        (0.675, -0.8, +0.3),
        (0.225, +0.8, -0.3),
        (0.05, +0.8, +0.3),
    ],
    "new_share_no_ppol" : [
        (0.5, -0.8, 0.0),
        (0.5, +0.8, 0.0),
    ],
    "old_share_no_ppol" : [
        (0.725, -0.8, 0.0),
        (0.275, +0.8, 0.0),
    ],
    "equal_share_high_ppol" : [
        (0.25, -0.8, -0.6),
        (0.25, -0.8, +0.6),
        (0.25, +0.8, -0.6),
        (0.25, +0.8, +0.6),
    ],
    "equal_share_low_ppol" : [
        (0.25, -0.8, -0.3),
        (0.25, -0.8, +0.3),
        (0.25, +0.8, -0.3),
        (0.25, +0.8, +0.3),
    ],
    "no_pol" : [
        (1.0, 0.0, 0.0),
    ],
}

In [5]:
workspaces = {}
fit_results = {}
for oo_name in oo_names:
    ws = {}
    fs = {}
    for name, run_conf in run_configurations.items():
        for run_ab in lumi_list:
            run_name = f"{name}_{run_ab}ab"
            w = build_sim_ws(run_conf, input_path, oo_name, run_ab, split_helicity_reversal=split_hel, pol_constraint=pol_constraint)
            model = w.pdf("sim_model")
            # model.Print("t")
            # print("test")
            ds = ROOT.RooStats.AsymptoticCalculator.GenerateAsimovData(model, w.set("observables"))
            fit_res = fit(w, "sim_model", ds, silent=True, constant_parameters=constant_parameters)
            ws[run_name] = w
            fs[run_name] = fit_res
    workspaces[oo_name] = ws
    fit_results[oo_name] = fs

[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_e_pol_run0
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooConstVar::0
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooConstVar::0.002
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_p_pol_run0
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooConstVar::0.0015
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_e_pol_run1
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_p_pol_run1
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_e_pol_run2
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_p_pol_run2
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constraint_e_pol_run3
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooGaussian::constr

In [6]:
plotter = Plotter()

In [7]:
legend_name_dict_oo = {
    "reco": "Reco, E-scheme",
    "mlvec_reco": "Reco, P-scheme",
    "clean_reco": "Reco, E-scheme, BIB-removal",
    "mlvec_clean_reco": "Reco, P-scheme, BIB-removal",
    "clean_brems_reco": "Reco, E-scheme, BIB-removal + brems",
    "mlvec_clean_brems_reco": "Reco, P-scheme, BIB-removal + brems",
    "cheat_clean_reco": "Reco, E-scheme, cheat BIB-removal",
    "mlvec_cheat_clean_reco": "Reco, P-scheme, cheat BIB-removal",
    "av_nurec_nomb_mc": "MC",
    "av_nurec_nomb_mc_cc10": "MC WW only",
    "av_nurec_nomb_mc_rlep": "MC + Reco lep",
    "av_nurec_nomb_mc_rlep_gamma": "MC + Reco lep, iso-#gamma brems",
    "av_nurec_nomb_mc_rlep_brems": "MC + Reco lep, window brems",
    "av_nurec_nomb_mc_rlep_cheated_brems": "MC + Reco lep, cheated brems",
}
name_dict_run_confs = {
    "no_pol": "P(0,0)",
    "new_share_high_ppol": "P(#pm0.8,#pm0.6), (5%,45%,45%,5%)",
    "old_share_high_ppol": "P(#pm0.8,#pm0.6), (5%,67.5%,22.5%,5%)",
    "new_share_low_ppol": "P(#pm0.8,#pm0.3), (5%,45%,45%,5%)",
    "old_share_low_ppol": "P(#pm0.8,#pm0.3), (5%,67.5%,22.5%,5%)",
    "new_share_no_ppol": "P(#pm0.8,0), (50%,50%)",
    "old_share_no_ppol": "P(#pm0.8,0), (72.5%,27.5%)",
    "equal_share_high_ppol": "P(#pm0.8,#pm0.6), (25%,25%,25%,25%)",
    "equal_share_low_ppol": "P(#pm0.8,#pm0.3), (25%,25%,25%,25%)",
}
legend_name_dict_oo |= {f"{name}_{run_ab}ab": f"{text}, {run_ab} ab^{{-1}}" for run_ab in lumi_list for name, text in name_dict_run_confs.items()}

In [ ]:
# for checks
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mc", "no_pol_3ab", fit_results, oo_names=[
    "mc",
    "av_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo)

Info in <TCanvas::Print>: pdf file test.pdf has been created


In [9]:
# mlvec and BIB-removal
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_mlvec_or_not", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    # "mlvec_clean_brems_reco",
    # "clean_brems_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_mlvec_or_not", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    # "mlvec_clean_brems_reco",
    # "clean_brems_reco",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_mlvec_or_not_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_mlvec_or_not.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_mlvec_or_not_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_mlvec_or_not.pdf has been created


In [10]:
# brems recovery
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_brems_reco", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_brems_reco", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_brems_mc", "no_pol_3ab", fit_results, oo_names=[
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_brems_mc", "no_pol_3ab", fit_results, oo_names=[
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_brems_reco_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_brems_reco.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_brems_reco_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_brems_reco.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_brems_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_brems_mc.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_brems_mc_no_legend.pdf has been created
Info in <TCanvas::Print>

In [11]:
# reco to mc
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_reco_to_mc", "no_pol_3ab", fit_results, oo_names=[
    "reco",
    "clean_brems_reco",
    # "mlvec_clean_brems_reco",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_reco_to_mc", "no_pol_3ab", fit_results, oo_names=[
    "reco",
    "clean_brems_reco",
    # "mlvec_clean_brems_reco",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_reco_to_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_reco_to_mc.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_reco_to_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_reco_to_mc.pdf has been created


In [12]:
# TODO
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_big_comp_no_pars", "clean_brems_reco", fit_results, parameter_names=[], legend_pars=(0.,0.,1.,1.), legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_high_ppol_3ab",
                                        "old_share_high_ppol_3ab",
                                        "new_share_low_ppol_3ab",
                                        "old_share_low_ppol_3ab",
                                        "new_share_no_ppol_3ab",
                                        "old_share_no_ppol_3ab",
                                        "equal_share_high_ppol_3ab",
                                        "equal_share_low_ppol_3ab",
                                        "no_pol_3ab",
                                    ], plot_dir=plot_dir)
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_big_comp_TGC", "clean_brems_reco", fit_results, parameter_names=["g1z", "ka", "la", ], legend_pars=(0.7,0.55,1.,1.), legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_high_ppol_3ab",
                                        "old_share_high_ppol_3ab",
                                        "new_share_low_ppol_3ab",
                                        "old_share_low_ppol_3ab",
                                        "new_share_no_ppol_3ab",
                                        "old_share_no_ppol_3ab",
                                        "equal_share_high_ppol_3ab",
                                        "equal_share_low_ppol_3ab",
                                        "no_pol_3ab",
                                    ], plot_dir=plot_dir)
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_big_comp_mW", "clean_brems_reco", fit_results, parameter_names=["mW", ], legend_pars=(0.7, 0., 1., 0.45),
                                    legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_high_ppol_3ab",
                                        "old_share_high_ppol_3ab",
                                        "new_share_low_ppol_3ab",
                                        "old_share_low_ppol_3ab",
                                        "new_share_no_ppol_3ab",
                                        "old_share_no_ppol_3ab",
                                        "equal_share_high_ppol_3ab",
                                        "equal_share_low_ppol_3ab",
                                        "no_pol_3ab",
                                    ], plot_dir=plot_dir)
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_big_comp_pols", "clean_brems_reco", fit_results, parameter_names=["e_pol_L", "e_pol_R", "p_pol_L", "p_pol_R"], legend_pars=(0.33,0.52,0.59,1.),
                                    legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_high_ppol_3ab",
                                        "old_share_high_ppol_3ab",
                                        "new_share_low_ppol_3ab",
                                        "old_share_low_ppol_3ab",
                                        "new_share_no_ppol_3ab",
                                        "old_share_no_ppol_3ab",
                                        "equal_share_high_ppol_3ab",
                                        "equal_share_low_ppol_3ab",
                                        "no_pol_3ab",
                                    ], plot_dir=plot_dir)

Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Warning in <TH1::TH1>: nbins is <=0 - set to nbins = 1
Error in <TCanvas::Range>: illegal world coordinates range: x1=0.000000, y1=-0.226667, x2=0.000000, y2=1.106667
Error in <TCanvas::RangeAxis>: illegal axis coordinates range: xmin=0.000000, ymin=0.000000, xmax=0.000000, ymax=1.000000
Error in <TCanvas::Range>: illegal world coordinates range: x1=0.000000, y1=-0.121951, x2=0.000000, y2=1.097561
Error in <TCanvas::RangeAxis>: illegal axis coordinates range: xmin=0.000000, ymin=0.000000, xmax=0.000000, ymax=1.000000
Info in <TCanvas::Print>: pdf file pl

In [13]:
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_TGC_ILC_LCF_FCC", "clean_brems_reco", fit_results, parameter_names=["g1z", "ka", "la", ], legend_pars=(0.52,0.67,1.,1.), legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_low_ppol_2ab",
                                        "new_share_high_ppol_3ab",
                                        "no_pol_11.2ab",
                                    ], plot_dir=plot_dir)
plotter.draw_plots_runs_per_oo_name("clean_brems_reco_mW_ILC_LCF_FCC", "clean_brems_reco", fit_results, parameter_names=["mW", ], #legend_pars=(0.7, 0., 1., 0.45),
                                    legend_name_dict=legend_name_dict_oo,
                                    run_names=[
                                        "new_share_low_ppol_2ab",
                                        "new_share_high_ppol_3ab",
                                        "no_pol_11.2ab",
                                    ], plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/clean_brems_reco_TGC_ILC_LCF_FCC_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/clean_brems_reco_TGC_ILC_LCF_FCC.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/clean_brems_reco_mW_ILC_LCF_FCC_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/clean_brems_reco_mW_ILC_LCF_FCC.pdf has been created


In [14]:
# cc10 vs cc20
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_mc_cc10_vs_cc20", "no_pol_3ab", fit_results, oo_names=[
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_cc10",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.58, 0.27), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_mc_cc10_vs_cc20", "no_pol_3ab", fit_results, oo_names=[
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_cc10",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.58, 0.27), plot_dir=plot_dir)
# reco
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_TGC_reco_cc10_vs_cc20", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_clean_brems_reco",
    # "mlvec_clean_brems_reco_san",
    "mlvec_clean_brems_reco_cc10",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.58, 0.27), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_3ab_mW_reco_cc10_vs_cc20", "no_pol_3ab", fit_results, oo_names=[
    "mlvec_clean_brems_reco",
    "mlvec_clean_brems_reco_cc10",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.58, 0.27), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_mc_cc10_vs_cc20_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_mc_cc10_vs_cc20.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_mc_cc10_vs_cc20_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_mc_cc10_vs_cc20.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_reco_cc10_vs_cc20_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_TGC_reco_cc10_vs_cc20.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_multi_run_split_hel_pol_constraint/no_pol_3ab_mW_reco_cc10_vs_cc20_no_legen

In [15]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_jm", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco_jm",
    "reco_jm",
    "mlvec_clean_reco_jm",
    "clean_reco_jm",
    "mlvec_cheat_clean_reco_jm",
    "cheat_clean_reco_jm",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_jm_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_reco_jm",
    "mlvec_clean_reco",
    "mlvec_clean_reco_jm",
    "mlvec_cheat_clean_reco",
    "mlvec_cheat_clean_reco_jm",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])

KeyError: 'new_share_high_ppol'

In [ ]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_mc_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_clean_reco",
    "mlvec_clean_brems_reco",
    "mlvec_cheat_clean_reco",
    # "kinfit_clean_reco",
    "av_nurec_post94_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc",
    "av_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_mW_reco_mc_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_clean_reco",
    "mlvec_clean_brems_reco",
    "mlvec_cheat_clean_reco",
    # "kinfit_clean_reco",
    "av_nurec_post94_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc",
    "av_mc",
], parameter_names=[
    "mW",
])

In [ ]:
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_mc", "no_pol", fit_results, oo_names=[
    "av_nomb_mc",
    "av_mc",
    "nomb_mc",
    "mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])

In [ ]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_mW_ultracheat", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
    "mW",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_pols_ultracheat", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "mc",
], parameter_names=[
    "e_pol_L",
    "e_pol_R",
    "p_pol_L",
    "p_pol_R",
])

In [ ]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_mW_rlepcomp", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep",
], parameter_names=[
    "g1z",
    "ka",
    "la",
    "mW",
])